# DS 310 — Homework 3: Indexing in Practice

**Name:** ________________________

---

## Overview

In this homework you run real experiments on a database with **500,000 rows** and observe how indexes affect query performance.

- **Q3.1** — Write four queries and time them with **no extra index** (auto-graded), then explain the result (written, 4 pts).
- **Q3.2** — Add three indexes (auto-graded), re-time the queries, and explain the speedups (written, 4 pts).
- **Q3.3** — Observe the write cost of indexes and explain the trade-off (written, 4 pts).
- **Q4** — Implement the core operations of a **B+ Tree** (auto-graded), observe how it scales, and explain why (written, 4 pts).
- **Q5** — One multiple-choice question about your use of AI tools (ungraded, 0 pts).

**Where you write code:** the four queries in **Q3.1**, the three `CREATE INDEX` statements in **Q3.2**, and the four marked blanks in the **Q4** B+ Tree. Each spot is marked **`YOUR CODE HERE`** (`--` in SQL, `#` in Python) — replace the marker with your code. Everything else is provided — just run it.

**File needed:** place `applications_large.db` in the **same folder** as this notebook.

When you finish: *Kernel → Restart & Run All*, confirm the checks pass, and upload **this `.ipynb`** to Gradescope.

## Setup — Run This First

### Install Otter for the self-checks (one time)

The `grader.check(...)` cells in this notebook use **Otter-Grader** to give you instant
feedback on your answers. Install it once by running the cell below (or run
`pip install otter-grader` in a terminal), then **restart the kernel** so the Setup cell
can find it.

If Otter (or a test) is not available on your machine, the notebook still works — the
check cells simply print a note, and your official grade is computed on Gradescope when
you submit.

In [ ]:
# Run this cell ONCE to install Otter for the self-checks, then restart the kernel.
# (Safe to re-run: if Otter is already installed, nothing changes.)
%pip install -q otter-grader

In [ ]:
# Setup — run this cell FIRST. Do NOT edit.
import sqlite3, pandas as pd, time, os, sys, math

# --- Otter self-check shim. The official grade is produced on Gradescope. ---
# Locally, grader.check runs only if Otter and the hidden tests are present;
# otherwise it prints a note and the notebook keeps running.
class _Grader:
    def __init__(self):
        self._nb = None
        try:
            import otter
            self._nb = otter.Notebook()
        except Exception:
            pass
    def check(self, name):
        if self._nb is None:
            print(f"[{name}] Otter is not installed (see the install cell at the top), "
                  "so this self-check was skipped - it will run on Gradescope.")
            return
        try:
            return self._nb.check(name)
        except Exception as e:
            print(f"[{name}] self-check unavailable locally ({type(e).__name__}); it will run on Gradescope.")
    def check_all(self):
        if self._nb is None:
            return
        try:
            return self._nb.check_all()
        except Exception as e:
            print("check_all unavailable locally:", e)
    def export(self, *a, **k):
        if self._nb is None:
            return
        try:
            return self._nb.export(*a, **k)
        except Exception:
            pass
grader = _Grader()

# --- Connect to the database (provided; place it next to this notebook) ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import files
    if not os.path.exists('applications_large.db'):
        print("Upload applications_large.db below.")
        files.upload()
    db_path = 'applications_large.db'
else:
    db_path = os.path.join(os.getcwd(), 'applications_large.db')
    if not os.path.exists(db_path):
        print(f"WARNING: applications_large.db not found at {db_path}.")
        print("Place applications_large.db in the same folder as this notebook.")

conn = sqlite3.connect(db_path)

def run(sql):
    """Run a query and return a DataFrame (empty DataFrame on error)."""
    try:
        return pd.read_sql_query(sql, conn)
    except Exception as e:
        print("Query error:", e)
        return pd.DataFrame()

def time_query(sql, runs=5):
    """Average execution time in milliseconds. Returns nan if the query is not valid yet."""
    try:
        conn.execute(sql).fetchall()  # warm-up
        t0 = time.perf_counter()
        for _ in range(runs):
            conn.execute(sql).fetchall()
        t1 = time.perf_counter()
        return (t1 - t0) / runs * 1000
    except Exception as e:
        print("Could not time query (is it written correctly?):", e)
        return float('nan')

def show_plan(sql):
    """Show SQLite's query execution plan."""
    try:
        for row in conn.execute(f"EXPLAIN QUERY PLAN {sql}").fetchall():
            print(row[3])
    except Exception as e:
        print("Plan error:", e)

# Verify the connection
try:
    n = conn.execute('SELECT COUNT(*) FROM Application').fetchone()[0]
    print("Setup complete. Connected to:", db_path)
    print(f"Rows in Application: {n:,}")
except Exception as e:
    print("Setup error:", e)


## The Data

The `Application` table has 500,000 rows:
```
Application(application_id, study_id, participant_id, submitted_at, status)
```

In [ ]:
display(run("SELECT * FROM Application LIMIT 8"))
display(run("SELECT status, COUNT(*) AS n FROM Application GROUP BY status"))

---
## Q3.1 — Queries Without an Index

First we time four queries with **no indexes** (other than the automatic primary-key index on `application_id`).

**Write the four queries** in the cell below, then run the timing cell and record the times.

In [ ]:
# Confirm no extra indexes exist yet (other than the automatic primary-key index).
print("Current indexes:")
display(run("SELECT name, tbl_name FROM sqlite_master WHERE type='index'"))

In [ ]:
# Q3.1 - Write the four queries the experiment will time.
# Each must be a SQL string. Do NOT change the variable names.

# Point lookup: return the FULL row for the application with application_id = 250000.
q_point  = """
-- YOUR CODE HERE
"""

# Status filter: return the COUNT of applications whose status is 'completed'.
q_status = """
-- YOUR CODE HERE
"""

# Participant lookup: return ALL rows for participant_id = 42.
q_ptcp   = """
-- YOUR CODE HERE
"""

# Date range: return the COUNT of applications submitted between
# '2024-01-01' and '2024-06-01' (inclusive).
#   Hint: you can write the range as  submitted_at >= ... AND submitted_at <= ...
#   or with SQL's  submitted_at BETWEEN ... AND ...  (BETWEEN is inclusive).
q_range  = """
-- YOUR CODE HERE
"""

In [ ]:
grader.check("q_point")

In [ ]:
grader.check("q_status")

In [ ]:
grader.check("q_ptcp")

In [ ]:
grader.check("q_range")

In [ ]:
# Do NOT edit. Time the four queries WITHOUT additional indexes. Record the results.
print("=== Query Times WITHOUT additional indexes ===")
t1 = time_query(q_point);  print(f"Point lookup   (application_id = 250000):        {t1:.1f} ms")
t2 = time_query(q_status); print(f"Status filter  (status = 'completed'):            {t2:.1f} ms")
t3 = time_query(q_ptcp);   print(f"Participant    (participant_id = 42):             {t3:.1f} ms")
t4 = time_query(q_range);  print(f"Date range     (2024-01-01 to 2024-06-01):        {t4:.1f} ms")

**Q3.1 — Written answer (manually graded, worth 4 points):**

The point lookup on `application_id` is much faster than the others even without an explicit index. Why?

*Your answer here:*

---
## Q3.2 — Queries With Indexes

Now **add indexes** on `status`, `participant_id`, and `submitted_at`. Write the `CREATE INDEX` statements below, then re-time the same queries.

The SQL statement for creating an index has this form:

```sql
CREATE INDEX index_name ON TableName(column);
```

In [ ]:
# Q3.2 - Add three indexes to the Application table, then commit.
conn.executescript("""
-- Create an index named idx_status      on Application(status)
-- Create an index named idx_participant on Application(participant_id)
-- Create an index named idx_date        on Application(submitted_at)

-- YOUR CODE HERE

""")
conn.commit()

print("Indexes now on Application:")
display(run("SELECT name, tbl_name FROM sqlite_master WHERE type='index'"))

In [ ]:
grader.check("indexes")

In [ ]:
# Do NOT edit. Re-time the same queries WITH indexes, then compute the speedup.
print("=== Query Times WITH indexes ===")
t1b = time_query(q_point);  print(f"Point lookup   (application_id = 250000):        {t1b:.1f} ms")
t2b = time_query(q_status); print(f"Status filter  (status = 'completed'):            {t2b:.1f} ms")
t3b = time_query(q_ptcp);   print(f"Participant    (participant_id = 42):             {t3b:.1f} ms")
t4b = time_query(q_range);  print(f"Date range     (2024-01-01 to 2024-06-01):        {t4b:.1f} ms")

print()
print("=== Speedup ===")
for label, before, after in [
    ("Point lookup ", t1, t1b),
    ("Status filter", t2, t2b),
    ("Participant  ", t3, t3b),
    ("Date range   ", t4, t4b),
]:
    if before is None or after is None or math.isnan(before) or math.isnan(after):
        print(f"{label}: n/a (write the query first)")
        continue
    s = before / after if after > 0.01 else float('inf')
    print(f"{label}: {s:.1f}x faster")

**Q3.2 — Written answers (manually graded, worth 4 points total):**

**A.** Which query saw the largest speedup? Why does indexing help especially for that query?

**B.** The `status` index gave a smaller speedup than the `participant_id` index. `status` has only 4 distinct values, while `participant_id` has thousands. How does this difference (selectivity) explain the gap in speedup?

*Your answers here:*

---
## Q3.3 — The Cost of Indexes on Writes

Indexes speed up reads, but every `INSERT` must also update all indexes. Run the cell below to measure this.

In [ ]:
# Do NOT edit. Measure the cost of indexes on writes.
import random
from datetime import date, timedelta

def insert_batch(n=1000):
    base = date(2025, 1, 1)
    rows = [(
        2000000 + i,
        random.randint(200, 400),
        random.randint(1, 5000),
        str(base + timedelta(days=random.randint(0, 365))),
        random.choice(['completed', 'pending', 'approved', 'withdrawn'])
    ) for i in range(n)]
    t0 = time.perf_counter()
    conn.executemany("INSERT INTO Application VALUES (?,?,?,?,?)", rows)
    conn.commit()
    t1 = time.perf_counter()
    conn.execute("DELETE FROM Application WHERE application_id >= 2000000")
    conn.commit()
    return (t1 - t0) * 1000

# Save the definitions of the indexes you created in Q3.2 so they can be restored
# after this experiment (the autograder checks them AFTER the whole notebook runs).
_index_defs = [r[0] for r in conn.execute(
    "SELECT sql FROM sqlite_master WHERE type='index' AND tbl_name='Application' AND sql IS NOT NULL")]

t_with = insert_batch()
print(f"Insert 1,000 rows WITH indexes:    {t_with:.1f} ms")

# Drop the indexes temporarily and repeat.
conn.execute("DROP INDEX IF EXISTS idx_status")
conn.execute("DROP INDEX IF EXISTS idx_participant")
conn.execute("DROP INDEX IF EXISTS idx_date")
conn.commit()

t_without = insert_batch()
print(f"Insert 1,000 rows WITHOUT indexes: {t_without:.1f} ms")
if t_without > 0:
    print(f"\nIndexes make inserts {t_with/t_without:.1f}x slower")

# Restore the indexes exactly as you created them in Q3.2.
for _sql in _index_defs:
    conn.execute(_sql)
conn.commit()
print("(Your indexes have been restored.)")

**Q3.3 — Written answer (manually graded, worth 4 points):**

Based on what you observed, describe the **trade-off** involved in adding indexes to a table. In what situation would you choose **not** to add an index, even on a frequently queried column?

*Your answer here:*

---
## Q4 — B+ Tree

Implement the missing lines in the B+ Tree below. Each spot is marked `# YOUR CODE HERE` — replace the marker (and any placeholder value on that line) with your code.

**A B+ Tree of order m:** internal nodes hold at most 2m keys, all data lives in leaf nodes, and leaves are linked in sorted order for fast range scans. No deletion required.

After each blank: run the small test cell right below it and **compare its output to the expected result** shown above it, then run the `grader.check(...)` cell to self-check.

### Setup — Node structure and tree skeleton (provided)

In [ ]:
class BPlusNode:
    def __init__(self, is_leaf=False):
        self.keys     = []
        self.is_leaf  = is_leaf
        self.values   = []       # leaf nodes only
        self.children = []       # internal nodes only
        self.next     = None     # pointer to next leaf


class BPlusTree:
    def __init__(self, order=2):
        self.order = order
        self.root  = BPlusNode(is_leaf=True)

    def _find_leaf(self, key):
        """Navigate from root to the leaf that would contain key."""
        node = self.root
        while not node.is_leaf:
            i = 0
            while i < len(node.keys) and key >= node.keys[i]:
                i += 1
            node = node.children[i]
        return node

    def height(self):
        h, node = 0, self.root
        while not node.is_leaf:
            h += 1
            node = node.children[0]
        return h

    def __repr__(self):
        lines = []
        self._print(self.root, 0, lines)
        return "\n".join(lines)

    def _print(self, node, depth, lines):
        indent = "  " * depth
        tag = "Leaf" if node.is_leaf else "Internal"
        lines.append(f"{indent}{tag} {node.keys}")
        if not node.is_leaf:
            for child in node.children:
                self._print(child, depth + 1, lines)

print("BPlusNode and BPlusTree skeleton defined")

### Q4.1 — `search`  *(worth 5 points)*

Find the leaf that should contain `key` (already done for you via `_find_leaf`), then return the value associated with `key` if it is present.

In [ ]:
def search(self, key):
    """Return the value for key, or None if not found."""
    leaf = self._find_leaf(key)
    for i, k in enumerate(leaf.keys):
        if k == key:
            pass  # YOUR CODE HERE - return the value stored at this position in the leaf
    return None

BPlusTree.search = search

**Run below.** A leaf with keys `[10, 20, 30]` and values `['a', 'b', 'c']`.

**Expected output:**
```
search(20) = b
search(99) = None
```

In [ ]:
_t = BPlusTree(order=2)
_t.root.keys = [10, 20, 30]
_t.root.values = ['a', 'b', 'c']

print(f"search(20) = {_t.search(20)}")
print(f"search(99) = {_t.search(99)}")

In [ ]:
grader.check("q4_search")

### Q4.2 — `range_search`  *(worth 5 points)*

Two blanks: what to collect for each matching key, and how to move to the next leaf using the linked-leaf list.

In [ ]:
def range_search(self, low, high):
    """Return list of (key, value) pairs where low <= key <= high."""
    results = []
    leaf = self._find_leaf(low)
    while leaf is not None:
        for i, k in enumerate(leaf.keys):
            if k > high:
                return results
            if k >= low:
                pass  # YOUR CODE HERE - append the matching (key, value) tuple to results
        leaf = None  # YOUR CODE HERE - follow the linked list to the next leaf
    return results

BPlusTree.range_search = range_search

**Run below.** Two linked leaves: `[10, 20] → [30, 40]` with values `['a','b'] → ['c','d']`.

**Expected output:**
```
range_search(15, 35) = [(20, 'b'), (30, 'c')]
```

In [ ]:
_t = BPlusTree(order=2)
leaf1 = BPlusNode(is_leaf=True); leaf1.keys=[10,20]; leaf1.values=['a','b']
leaf2 = BPlusNode(is_leaf=True); leaf2.keys=[30,40]; leaf2.values=['c','d']
leaf1.next = leaf2
_t.root = leaf1

print(f"range_search(15, 35) = {_t.range_search(15, 35)}")

In [ ]:
grader.check("q4_range")

### Insert logic (provided — read before Q4.3)

This drives the recursive insert and decides when a split is needed. You do not modify it — but Q4.3 and Q4.4 implement the two split routines it calls.

In [ ]:
def insert(self, key, value=None):
    result = self._insert_recursive(self.root, key, value)
    if result is not None:
        mid_key, new_node = result
        new_root = BPlusNode(is_leaf=False)
        new_root.keys     = [mid_key]
        new_root.children = [self.root, new_node]
        self.root = new_root

def _insert_recursive(self, node, key, value):
    if node.is_leaf:
        i = 0
        while i < len(node.keys) and key > node.keys[i]:
            i += 1
        node.keys.insert(i, key)
        node.values.insert(i, value)
        if len(node.keys) <= 2 * self.order:
            return None
        return self._split_leaf(node)
    else:
        i = 0
        while i < len(node.keys) and key >= node.keys[i]:
            i += 1
        result = self._insert_recursive(node.children[i], key, value)
        if result is None:
            return None
        mid_key, new_child = result
        node.keys.insert(i, mid_key)
        node.children.insert(i + 1, new_child)
        if len(node.keys) <= 2 * self.order:
            return None
        return self._split_internal(node)

BPlusTree.insert = insert
BPlusTree._insert_recursive = _insert_recursive
print("insert() defined")

### Q4.3 — `_split_leaf`  *(worth 5 points)*

Split a full leaf in half. The first key of the new (right) leaf is **copied** up to the parent — it must also remain in the leaf itself.

In [ ]:
def _split_leaf(self, leaf):
    """Split a full leaf. The first key of the new (right) leaf is COPIED up."""
    mid = len(leaf.keys) // 2
    new_leaf = BPlusNode(is_leaf=True)
    new_leaf.keys   = []           # YOUR CODE HERE - right half of leaf.keys
    new_leaf.values = []           # YOUR CODE HERE - right half of leaf.values
    leaf.keys       = leaf.keys    # YOUR CODE HERE - keep only the left half
    leaf.values     = leaf.values  # YOUR CODE HERE - keep only the left half
    new_leaf.next   = leaf.next
    leaf.next       = new_leaf
    # (The guard below just keeps the notebook running before you fill in the blanks.)
    return (new_leaf.keys[0] if new_leaf.keys else None), new_leaf

BPlusTree._split_leaf = _split_leaf

**Run below.** A full leaf with keys `[10, 20, 30, 40]`.

**Expected output:**
```
left leaf keys:  [10, 20]
right leaf keys: [30, 40]
mid_key (copied up): 30
left.next is right leaf: True
```

In [ ]:
_leaf = BPlusNode(is_leaf=True)
_leaf.keys = [10, 20, 30, 40]
_leaf.values = ['a','b','c','d']
_t = BPlusTree(order=2)

mid_key, new_leaf = _t._split_leaf(_leaf)
print(f"left leaf keys:  {_leaf.keys}")
print(f"right leaf keys: {new_leaf.keys}")
print(f"mid_key (copied up): {mid_key}")
print(f"left.next is right leaf: {_leaf.next is new_leaf}")

In [ ]:
grader.check("q4_split_leaf")

### Q4.4 — `_split_internal`  *(worth 5 points)*

Split a full internal node. The middle key is **pushed** up to the parent — unlike a leaf split, it is removed from this node.

In [ ]:
def _split_internal(self, node):
    """Split a full internal node. The middle key is PUSHED up (removed from this node)."""
    mid     = len(node.keys) // 2
    mid_key = None  # YOUR CODE HERE - which key is pushed up to the parent?
    new_node = BPlusNode(is_leaf=False)
    new_node.keys     = node.keys[mid + 1:]
    new_node.children = node.children[mid + 1:]
    node.keys         = node.keys[:mid]
    node.children     = node.children[:mid + 1]
    return mid_key, new_node

BPlusTree._split_internal = _split_internal

**Run below.** A full internal node with keys `[10, 20, 30, 40]` and children `['c0','c1','c2','c3','c4']`.

**Expected output:**
```
left node keys:     [10, 20]
right node keys:    [40]
mid_key (pushed up): 30
left children:  ['c0', 'c1', 'c2']
right children: ['c3', 'c4']
```

In [ ]:
_node = BPlusNode(is_leaf=False)
_node.keys = [10, 20, 30, 40]
_node.children = ['c0','c1','c2','c3','c4']
_t = BPlusTree(order=2)

mid_key, new_node = _t._split_internal(_node)
print(f"left node keys:     {_node.keys}")
print(f"right node keys:    {new_node.keys}")
print(f"mid_key (pushed up): {mid_key}")
print(f"left children:  {_node.children}")
print(f"right children: {new_node.children}")

In [ ]:
grader.check("q4_split_internal")

### Full tree — putting it all together

Once all four blanks above produce the expected output, run this cell to build a complete tree and see its full structure.

**Expected output (order m=2, after inserting all 16 keys):**
```
Internal [61]
  Internal [20, 40]
    Leaf [3, 5, 10, 13]
    Leaf [20, 28, 34]
    Leaf [40, 51]
  Internal [77, 90]
    Leaf [61, 68]
    Leaf [77, 80]
    Leaf [90, 93, 97]

Height: 2
```

In [ ]:
t = BPlusTree(order=2)
try:
    for k in [20, 40, 68, 90, 3, 10, 28, 34, 51, 61, 77, 80, 93, 97, 13, 5]:
        t.insert(k, f"val_{k}")
    print(t)
    print(f"\nHeight: {t.height()}")
except Exception as e:
    print("The tree could not be built yet - finish Q4.1-Q4.4 first.")
    print(f"({type(e).__name__}: {e})")

**Run below to check `search` and `range_search` on the full tree.**

**Expected output:**
```
search(51) = val_51
search(3)  = val_3
search(99) = None
range_search(30, 70) = [(34, 'val_34'), (40, 'val_40'), (51, 'val_51'), (61, 'val_61'), (68, 'val_68')]
```

In [ ]:
try:
    print(f"search(51) = {t.search(51)}")
    print(f"search(3)  = {t.search(3)}")
    print(f"search(99) = {t.search(99)}")
    print(f"range_search(30, 70) = {t.range_search(30, 70)}")
except Exception as e:
    print("Finish Q4.1-Q4.4 first.", f"({type(e).__name__}: {e})")

### Timing experiment

Insert 100, then 1,000, then 10,000 random keys, and time 1,000 searches at each size. (The plots need `matplotlib`; if it is not installed the printed numbers are enough.)

**Reading the plots:** the x-axis is **logarithmic** — each tick is 10x the previous one. On such an axis, a quantity that grows like log(n) rises by a **constant amount** per tick, so it appears as a **straight line**. Keep that in mind when you answer the written questions below.

In [ ]:
# Do NOT edit. Timing experiment: how does search scale with n?
import random, time
try:
    import matplotlib.pyplot as plt
    _HAVE_MPL = True
except ImportError:
    _HAVE_MPL = False   # the plots are optional; the printed numbers are what matters

sizes = [100, 1000, 10000]
avg_times, heights = [], []
try:
    for n in sizes:
        tree = BPlusTree(order=10)
        keys = random.sample(range(n * 10), n)
        for k in keys:
            tree.insert(k, k)
        queries = [random.choice(keys) for _ in range(1000)]
        t0 = time.perf_counter()
        for q in queries:
            tree.search(q)
        us = (time.perf_counter() - t0) / 1000 * 1e6
        avg_times.append(us)
        heights.append(tree.height())
        print(f"n={n:>7,}  height={tree.height()}  avg search={us:.2f} us")
    if _HAVE_MPL:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        ax1.plot(sizes, avg_times, 'o-', linewidth=2, markersize=8)
        ax1.set_xscale('log'); ax1.set_xlabel("n"); ax1.set_ylabel("microseconds")
        ax1.set_title("Search time vs n"); ax1.grid(alpha=0.3)
        ax2.plot(sizes, heights, 's-', linewidth=2, markersize=8)
        ax2.set_xscale('log'); ax2.set_xlabel("n"); ax2.set_ylabel("height")
        ax2.set_title("Tree height vs n"); ax2.grid(alpha=0.3)
        plt.tight_layout(); plt.show()
except Exception as e:
    print("The timing experiment needs Q4.1-Q4.4 finished.", f"({type(e).__name__}: {e})")

**Q4 — Written answers (manually graded, worth 4 points total):**

**A.** How does search time change as n grows from 100 to 10,000? Is this consistent with O(log n)?

**B.** Why does a B+ tree stay shallow even with 10,000 keys?

*Your answers here:*

---
## Q5 — Use of AI tools (ungraded)

This question is **not graded** and has no effect on your score — please answer honestly.
It helps us understand how AI tools are being used in the course.

**Q5.1 — Which of the following describe how you used an AI assistant (ChatGPT, Claude, Copilot, ...) on this assignment?**

**Select all that apply** — you may choose more than one option. Write the letter(s) below.

- **(a)** Not at all
- **(b)** To check my work
- **(c)** To explain my mistakes
- **(d)** To generate answers

*Your answer (one or more of a, b, c, d):*


---
## Submit

Run **all** cells top to bottom (*Kernel → Restart & Run All*), make sure every check passes, then upload **this notebook** (`.ipynb`) to Gradescope. The written answers (Q3.1, Q3.2, Q3.3, Q4) are graded by hand — 4 points each, 16 points total.

In [ ]:
grader.check_all()